# 03 Agentic App: End-to-End Support Workflow

This notebook demonstrates the complete CultPass support-ticket workflow using the real LangGraph orchestrator, OpenAI LLM, SQLite data, Chroma knowledge base, and DB/RAG tool logic. It uses in-process tool adapters so saved notebook execution is stable while the CLI app continues to load the same tools over MCP transport.

## Setup

In [1]:
from __future__ import annotations

import io
import json
import logging
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "solution" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from solution.agentic.tools.local_tools import load_local_tools
from solution.agentic.workflow import build_orchestrator
from solution.logging_config import (
    LOG_FORMAT,
    JsonlEventFormatter,
    StructuredEventFilter,
    configure_logging,
)

load_dotenv(PROJECT_ROOT / ".env")
configure_logging()

Out[0]: WindowsPath('C:/Users/rsurs/OneDrive/Documents/Root/project-4/logs/udahub.log')


In [2]:
log_stream = io.StringIO()
log_handler = logging.StreamHandler(log_stream)
log_handler.setFormatter(logging.Formatter(LOG_FORMAT))

structured_stream = io.StringIO()
structured_handler = logging.StreamHandler(structured_stream)
structured_handler.setFormatter(JsonlEventFormatter())
structured_handler.addFilter(StructuredEventFilter())

udahub_logger = logging.getLogger("udahub")
udahub_logger.setLevel(logging.INFO)
udahub_logger.addHandler(log_handler)
udahub_logger.addHandler(structured_handler)
udahub_logger.propagate = False

print("Timestamped and structured log capture is ready for udahub.* loggers.")

Timestamped and structured log capture is ready for udahub.* loggers.


In [3]:
tools = load_local_tools()
preflight_docs = await tools["knowledge_search"].ainvoke(
    {"query": "How do I reserve an event on CultPass?", "k": 3}
)
assert preflight_docs, "Preflight failed: knowledge_search returned no reservation docs. Run solution/scripts/build_index.py and rerun this notebook."
assert any(doc.get("title") == "How to Reserve a Spot for an Event" for doc in preflight_docs), preflight_docs
print(f"Knowledge preflight retrieved {len(preflight_docs)} docs: {[doc.get('title') for doc in preflight_docs]}")

orchestrator = build_orchestrator(tools)
print("Real orchestrator loaded with local DB/RAG tool adapters and OpenAI LLM.")

Knowledge preflight retrieved 3 docs: ['How to Reserve a Spot for an Event', 'How to Change a Reserved Event', "What's Included in a CultPass Subscription"]
Real orchestrator loaded with local DB/RAG tool adapters and OpenAI LLM.


## Trace Helper

In [4]:
async def run_ticket(ticket_id: str, customer_message: str) -> dict:
    log_stream.seek(0)
    log_stream.truncate(0)
    structured_stream.seek(0)
    structured_stream.truncate(0)

    config = {"configurable": {"thread_id": ticket_id}}
    state = await orchestrator.ainvoke(
        {"ticket_id": ticket_id, "messages": [HumanMessage(content=customer_message)]},
        config=config,
    )

    retrieved_docs = state.get("retrieved_docs") or []
    tool_results = state.get("tool_results") or []
    final_message = state["messages"][-1].content if state.get("messages") else ""
    logs = log_stream.getvalue().strip()
    structured_events = [
        json.loads(line)
        for line in structured_stream.getvalue().splitlines()
        if line.strip()
    ]

    print(f"TICKET: {ticket_id}")
    print(f"USER: {customer_message}")
    print(f"CLASSIFICATION: category={state.get('category')} priority={state.get('priority')} confidence={state.get('confidence')}")
    print(f"ROUTE: {state.get('route')}")
    print(f"RETRIEVED_DOCS: count={len(retrieved_docs)} titles={[doc.get('title') for doc in retrieved_docs]}")
    print("TOOL_RESULTS:")
    print(json.dumps(tool_results, indent=2, default=str) if tool_results else "[]")
    print(f"ESCALATION_REQUIRED: {state.get('escalation_required')}")
    print("FINAL_RESPONSE:")
    print(final_message)
    print("STRUCTURED_EVENTS:")
    print(json.dumps(structured_events, indent=2, default=str))
    print("LOGS:")
    print(logs)

    return {"state": state, "structured_events": structured_events}

## Scenario 1: Knowledge-Base Resolution

In [5]:
kb_result = await run_ticket(
    "demo-kb-resolution",
    "How do I reserve an event on CultPass?",
)
kb_state = kb_result["state"]
assert kb_state.get("route") == "resolver"
assert len(kb_state.get("retrieved_docs") or []) > 0
assert kb_state.get("escalation_required") is False
assert any(event.get("event") == "knowledge_search" and event.get("retrieval_count", 0) > 0 for event in kb_result["structured_events"])
assert any(event.get("event") == "finalized" and event.get("final_status") == "resolved" for event in kb_result["structured_events"])

TICKET: demo-kb-resolution
USER: How do I reserve an event on CultPass?
CLASSIFICATION: category=General priority=low confidence=0.9
ROUTE: resolver
RETRIEVED_DOCS: count=3 titles=['How to Reserve a Spot for an Event', 'How to Change a Reserved Event', "What's Included in a CultPass Subscription"]
TOOL_RESULTS:
[]
ESCALATION_REQUIRED: False
FINAL_RESPONSE:
You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping 'Reserve'. If it's a premium or limited event, check if reservation confirmation is required via email. Remember to arrive at least 15 minutes early with your QR code ready!
STRUCTURED_EVENTS:
[
  {
    "agent": "supervisor",
    "event": "memory_search",
    "level": "INFO",
    "logger": "udahub.supervisor",
    "retrieval_count": 1,
    "ticket_id": "demo-kb-resolution",
    "timestamp": "2026-06-17T10:51:39.200972+00:00",
    "tool_name": "memory_search",
    "tool_success": true
  },
  {
    "agent": "classifier",
    "category":

## Scenario 2: Tool-Assisted Account Lookup

In [6]:
tool_result = await run_ticket(
    "demo-tool-subscription",
    "What subscription do I have? My customer id is f556c0.",
)
tool_state = tool_result["state"]
assert tool_state.get("route") == "tool_agent"
assert tool_state.get("tool_results")
assert tool_state.get("escalation_required") is False
assert any(event.get("tool_name") == "get_subscription" and event.get("tool_success") is True for event in tool_result["structured_events"])
assert any(event.get("event") == "finalized" and event.get("final_status") == "resolved" for event in tool_result["structured_events"])

TICKET: demo-tool-subscription
USER: What subscription do I have? My customer id is f556c0.
CLASSIFICATION: category=Membership priority=medium confidence=0.9
ROUTE: tool_agent
RETRIEVED_DOCS: count=0 titles=[]
TOOL_RESULTS:
[
  {
    "tool": "get_subscription",
    "result": {
      "found": true,
      "user_id": "f556c0",
      "status": "active",
      "tier": "premium",
      "monthly_quota": 5
    }
  }
]
ESCALATION_REQUIRED: False
FINAL_RESPONSE:
You have an active **premium** subscription with a monthly quota of **5**. If you have any more questions or need assistance, feel free to ask!
STRUCTURED_EVENTS:
[
  {
    "agent": "supervisor",
    "event": "memory_search",
    "level": "INFO",
    "logger": "udahub.supervisor",
    "retrieval_count": 0,
    "ticket_id": "demo-tool-subscription",
    "timestamp": "2026-06-17T10:51:44.620147+00:00",
    "tool_name": "memory_search",
    "tool_success": true
  },
  {
    "agent": "classifier",
    "category": "Membership",
    "confiden

## Scenario 3: Escalation After Tool Usage

In [7]:
escalation_result = await run_ticket(
    "demo-refund-escalation",
    "Please approve a refund for customer a4ab87 for my last booking.",
)
escalation_state = escalation_result["state"]
assert escalation_state.get("route") == "tool_agent"
assert escalation_state.get("escalation_required") is True
assert any(event.get("tool_name") == "process_refund" for event in escalation_result["structured_events"])
assert any(event.get("event") == "finalized" and event.get("final_status") == "escalated" for event in escalation_result["structured_events"])

TICKET: demo-refund-escalation
USER: Please approve a refund for customer a4ab87 for my last booking.
CLASSIFICATION: category=Refund priority=medium confidence=0.9
ROUTE: tool_agent
RETRIEVED_DOCS: count=0 titles=[]
TOOL_RESULTS:
[
  {
    "tool": "lookup_customer",
    "result": {
      "found": true,
      "user_id": "Alice Kingsley",
      "email": "alice.kingsley@wonderland.com",
      "is_blocked": true,
      "subscription": {
        "status": "cancelled",
        "tier": "basic",
        "monthly_quota": 3
      }
    }
  },
  {
    "tool": "list_reservations",
    "result": {
      "user_id": "a4ab87",
      "count": 2,
      "reservation": [
        {
          "reservation_id": "346e63",
          "status": "reserved",
          "experience_title": "Carnival History Tour in Olinda",
          "when": "2026-06-14T14:33:44.619594",
          "is_premium": true
        },
        {
          "reservation_id": "4f8fb9",
          "status": "reserved",
          "experience_titl

## Scenario 4: Edge Case / Unknown Issue

In [8]:
edge_result = await run_ticket(
    "demo-edge-unknown",
    "Can CultPass arrange a private moon concert next week?",
)
edge_state = edge_result["state"]
assert edge_state.get("route") in {"resolver", "escalation"}
assert edge_state.get("escalation_required") is True
assert any(event.get("event") == "finalized" and event.get("final_status") == "escalated" for event in edge_result["structured_events"])

TICKET: demo-edge-unknown
USER: Can CultPass arrange a private moon concert next week?
CLASSIFICATION: category=General priority=low confidence=0.9
ROUTE: resolver
RETRIEVED_DOCS: count=3 titles=["What's Included in a CultPass Subscription", 'How to Reserve a Spot for an Event', 'How to View Upcoming Reservations']
TOOL_RESULTS:
[]
ESCALATION_REQUIRED: True
FINAL_RESPONSE:
Hi there! Thank you for reaching out to us about arranging a private moon concert. I understand how exciting this is, and I've escalated your request to a teammate who will follow up with you shortly. We appreciate your patience!
STRUCTURED_EVENTS:
[
  {
    "agent": "supervisor",
    "event": "memory_search",
    "level": "INFO",
    "logger": "udahub.supervisor",
    "retrieval_count": 1,
    "ticket_id": "demo-edge-unknown",
    "timestamp": "2026-06-17T10:51:59.382719+00:00",
    "tool_name": "memory_search",
    "tool_success": true
  },
  {
    "agent": "classifier",
    "category": "General",
    "confidence":